In [ ]:
#loading required packages
import re
import numpy as np
import tweepy 
from tweepy import OAuthHandler 
from textblob import TextBlob 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from wordcloud import WordCloud
from better_profanity import profanity
import seaborn
import sklearn
from sklearn.metrics import precision_score,recall_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import TFBertModel, BertTokenizer

main_path = r'C:\Users\Jerry\Desktop\Xacking PhD\dataset\codes\Sentiment_Analysis-master\Sentiment_Analysis-master'

In [ ]:
#reading the dataset
# df = pd.read_csv('ebola_tweets.csv')
df = pd.read_csv(main_path + '/ebola_tweets.csv')

In [ ]:
df.drop_duplicates(subset = ["timestamp", "user", "text"], inplace=True)
print(f"all tweets: {df.shape}")

In [ ]:
df.rename(columns={"text": "tweets"}, inplace=True)
df.head()

In [ ]:
# Converting only the tweets into a list
tweet_list = df.tweets.to_list()
# tweet_list

In [ ]:
import re
import html
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resources if not already
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tweet(tweet):
    if isinstance(tweet, float):
        return ""

    r = tweet.lower()
    r = profanity.censor(r)

    # Decode HTML entities (&amp; → &)
    r = html.unescape(r)

    # Remove mentions, hashtags, links
    r = re.sub(r"@[A-Za-z0-9_]+", "", r)
    r = re.sub(r"#[A-Za-z0-9_]+", "", r)
    r = re.sub(r"http\S+", "", r)

    # Remove punctuations and brackets
    r = re.sub(r"[()!?]", " ", r)
    r = re.sub(r"\[.*?\]", " ", r)

    # Remove contractions artifacts (don't → dont)
    r = re.sub(r"\'", "", r)

    # Keep only letters (remove digits)
    r = re.sub(r"[^a-z0-9]", " ", r)

    # Remove leftover "rt" completely (retweet marker)
    r = re.sub(r"\brt\b", "", r)

    # Tokenize
    r = r.split()

    # Remove stopwords + very short tokens
    r = [w for w in r if w not in stop_words and len(w) > 2]

    # Lemmatize remaining tokens
    # r = [lemmatizer.lemmatize(w) for w in r]

    return " ".join(r)


In [ ]:
#cleaning the data
cleaned = [clean_tweet(tw) for tw in tweet_list]

In [ ]:
# Defining the sentiment objects using TextBlob
sentiment_objects = [TextBlob(tweet) for tweet in cleaned]
sentiment_objects[0].polarity, sentiment_objects[0]

In [ ]:
# Creating a list of polarity values and tweet text
sentiment_values = [[tweet.sentiment.polarity, str(tweet)] for tweet in sentiment_objects]
#sentiment_values[0]
# sentiment_values[0:99]

In [ ]:
# Creating a dataframe of each tweet against its polarity
sentiment_df = pd.DataFrame(sentiment_values, columns=["polarity", "tweet"])
sentiment_df.head()

In [ ]:
sentiment_df['sentiment_class'] = np.where(sentiment_df['polarity'] > 0, 'Positive', np.where(sentiment_df['polarity'] <0, 'Negative', 'Neutral'))

In [ ]:
# add labeling
symptom_keywords = [
    "sore", "fever", "fatigue", "malaise", "muscle pain", "headache", "throat",
    "vomiting", "diarrhoea", "diarrhea", "abdominal pain", "rash",
    "kidney", "liver", "ill", "weak", "platelets", "rest", "sick", "fighting", "numb", "flu"
]

# Normalize to lowercase and handle multi-word phrases
symptom_keywords = [kw.lower() for kw in symptom_keywords]

# Function to label tweets
def label_tweet(text):
    text_lower = text.lower()
    return "Positive" if any(kw in text_lower for kw in symptom_keywords) else "Negative"

def count_symptom_keywords(text):
    text_lower = text.lower()
    return sum(1 for kw in symptom_keywords if kw in text_lower)


# Create DataFrame and label tweets labels are done by symptoms and category by sentiment analysis
sentiment_df['symptom_keyword_count'] = sentiment_df['tweet'].apply(count_symptom_keywords)
sentiment_df['label'] = sentiment_df['tweet'].apply(label_tweet)

In [ ]:
sentiment_df.head()

In [ ]:
# sentiment_df['category'].value_counts()
sentiment_df['label'].value_counts()

In [ ]:
#category and label encoding
mapping1 = {'Positive':0,'Negative':1,'Neutral':2}
mapping2 = {'Positive':1,'Negative':0}
sentiment_df['sentiment_class'] = sentiment_df['sentiment_class'].map(mapping1)
sentiment_df['label'] = sentiment_df['label'].map(mapping2)

In [ ]:
# Store
sentiment_df.to_csv(main_path + "/sentiment_df4.csv", index=False)
sentiment_df.shape

In [ ]:
#loading required packages
import re
import numpy as np
import tweepy 
from tweepy import OAuthHandler 
from textblob import TextBlob 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from wordcloud import WordCloud
from better_profanity import profanity
import seaborn
import sklearn
from sklearn.metrics import precision_score,recall_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import TFBertModel, BertTokenizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

main_path = r'C:\Users\Jerry\Desktop\Xacking PhD\dataset\codes\Sentiment_Analysis-master\Sentiment_Analysis-master'

In [ ]:
# Retrieve
preprocessed_df = pd.read_csv(main_path + "/sentiment_df4.csv")
preprocessed_df['tweet'] = preprocessed_df['tweet'].fillna("")
preprocessed_df.shape

In [ ]:
# Feature extraction using bag of words model
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(preprocessed_df['tweet'])

sequences = tokenizer.texts_to_sequences(preprocessed_df['tweet'])

padded_sequences = pad_sequences(sequences, padding='post', maxlen=None)

sentiment_class = preprocessed_df['sentiment_class']
symptom_keyword_count = preprocessed_df['symptom_keyword_count']
labels = preprocessed_df['label']

In [ ]:
padded_sequences.shape, sentiment_class.shape, symptom_keyword_count.shape, labels.shape

In [ ]:
# Define CNN for text
tweet_input = Input(shape=(padded_sequences.shape[1],))
embedding = Embedding(input_dim=1000, output_dim=50)(tweet_input)
# cnn = Conv1D(filters=32, kernel_size=3, activation='relu')(embedding)
# cnn = GlobalMaxPooling1D()(cnn)
# cnn_model = Model(inputs=tweet_input, outputs=cnn)

In [ ]:
from tensorflow.keras.layers import Concatenate

conv3 = Conv1D(64, 3, activation='relu')(embedding)
conv4 = Conv1D(64, 4, activation='relu')(embedding)
conv5 = Conv1D(64, 5, activation='relu')(embedding)

cnn = Concatenate()([
    GlobalMaxPooling1D()(conv3),
    GlobalMaxPooling1D()(conv4),
    GlobalMaxPooling1D()(conv5)
])
cnn_model = Model(inputs=tweet_input, outputs=cnn)

In [ ]:
# from tensorflow.keras.layers import LSTM
# # LSTM branch
# lstm = LSTM(64, return_sequences=False)(embedding)
# lstm_model = Model(inputs=tweet_input, outputs=lstm)

In [ ]:
from tensorflow.keras.layers import Bidirectional, LSTM

lstm = Bidirectional(
    LSTM(32, return_sequences=False)
)(embedding)
lstm_model = Model(inputs=tweet_input, outputs=lstm)

In [ ]:
# Extract Features
cnn_features = cnn_model.predict(padded_sequences)
lstm_features = lstm_model.predict(padded_sequences)

In [ ]:
cnn_features.shape, lstm_features.shape

In [ ]:
# Prepare auxiliary features
sentiment_class_feature = sentiment_class.values.reshape(-1, 1) 

In [ ]:
# Concatenate CNN, LSTM features with auxiliary features
combined_features = np.hstack([
    cnn_features,
    lstm_features,
    sentiment_class_feature
])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
combined_features = scaler.fit_transform(combined_features)

In [ ]:
train_data, test_data, train_labels, test_labels = train_test_split(combined_features, labels, test_size=0.2, random_state=0)

train_data, val_data, train_labels, val_labels = train_test_split(train_data, train_labels, test_size=0.2, random_state=0)

In [ ]:
# train a RF model
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(train_data, train_labels)

In [ ]:
y_pred = rf.predict(test_data)

In [ ]:
print("Classification Report:\n")
print(classification_report(test_labels, y_pred))

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# PCA
pca = PCA(n_components=2)
features_2d = pca.fit_transform(combined_features)

labels_np = np.array(labels)

plt.figure(figsize=(7, 6))

# Negative class (0)
idx_neg = labels_np == 0
plt.scatter(
    features_2d[idx_neg, 0],
    features_2d[idx_neg, 1],
    color='blue',
    label='Negative',
    alpha=0.6
)

# Positive class (1)
idx_pos = labels_np == 1
plt.scatter(
    features_2d[idx_pos, 0],
    features_2d[idx_pos, 1],
    color='red',
    label='Positive',
    alpha=0.6
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("PCA Projection of Combined Tweet Features")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

labels_np = np.array(labels)

# -----------------------------
# 1. PCA for CNN features only
# -----------------------------
pca_cnn = PCA(n_components=2)
cnn_2d = pca_cnn.fit_transform(cnn_features)

# ---------------------------------
# 2. PCA for CNN + LSTM features
# ---------------------------------
pca_cnn_lstm = PCA(n_components=2)
cnn_lstm_2d = pca_cnn_lstm.fit_transform(lstm_features)

# -----------------------------
# 3. Plot side by side
# -----------------------------
plt.figure(figsize=(14, 6))

# ---- CNN only ----
plt.subplot(1, 2, 1)

idx_neg = labels_np == 0
idx_pos = labels_np == 1

plt.scatter(
    cnn_2d[idx_neg, 0],
    cnn_2d[idx_neg, 1],
    color='blue',
    label='Negative',
    alpha=0.6
)

plt.scatter(
    cnn_2d[idx_pos, 0],
    cnn_2d[idx_pos, 1],
    color='red',
    label='Positive',
    alpha=0.6
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("PCA Projection: CNN Features")
plt.legend()

# ---- CNN + LSTM ----
plt.subplot(1, 2, 2)

plt.scatter(
    cnn_lstm_2d[idx_neg, 0],
    cnn_lstm_2d[idx_neg, 1],
    color='blue',
    label='Negative',
    alpha=0.6
)

plt.scatter(
    cnn_lstm_2d[idx_pos, 0],
    cnn_lstm_2d[idx_pos, 1],
    color='red',
    label='Positive',
    alpha=0.6
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("PCA Projection: LSTM Features")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Feature importance
import numpy as np
import matplotlib.pyplot as plt

importances = rf.feature_importances_
colors = plt.cm.viridis(importances / importances.max())

plt.bar(range(len(importances)), importances, color=colors)
plt.xlabel("Feature Index")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of features
n_cnn = cnn_features.shape[1]
n_lstm = lstm_features.shape[1]

# Feature importances from trained RF
importances = rf.feature_importances_

# Split importances
cnn_importances = importances[:n_cnn]
lstm_importances = importances[n_cnn:n_cnn + n_lstm]

# Colors
cnn_colors = plt.cm.Blues(cnn_importances / cnn_importances.max())
lstm_colors = plt.cm.Oranges(lstm_importances / lstm_importances.max())

# -----------------------------
# Plot side by side
# -----------------------------
plt.figure(figsize=(14, 5))

# CNN feature importance
plt.subplot(1, 2, 1)
plt.bar(range(n_cnn), cnn_importances, color=cnn_colors)
plt.xlabel("CNN Feature Index")
plt.ylabel("Importance")
plt.title("(A) Feature Importance (CNN)")

# LSTM feature importance
plt.subplot(1, 2, 2)
plt.bar(range(n_lstm), lstm_importances, color=lstm_colors)
plt.xlabel("LSTM Feature Index")
plt.ylabel("Importance")
plt.title("(B) Feature Importance (LSTM)")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(6, 5))
class_names = ["Negative", "Positive"]

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)


plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Precision–Recall (PR) Curve 
from sklearn.metrics import precision_recall_curve

y_prob = rf.predict_proba(test_data)[:,1]
precision, recall, _ = precision_recall_curve(test_labels, y_prob)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")


In [ ]:
# ROC
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(test_labels, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.legend()
